# Comparativa de espacios latentes

Carga los resultados guardados por `modelos/validacion.ipynb` y genera la comparativa.

---
## 0 — Entorno (Colab o local)

In [ ]:
import json, os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Detectar entorno ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
PROJECT_ROOT = "/content/drive/MyDrive/TFM/"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
DATA_ROOT = os.path.join(PROJECT_ROOT, "data", "fase1")
VIS_ROOT  = os.path.join(PROJECT_ROOT, "visualizations")
os.makedirs(VIS_ROOT, exist_ok=True)

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"VIS_ROOT  : {VIS_ROOT}")

---
## 1 — Cargar resultados guardados

In [ ]:
MODELS = ["sd15", "sd21", "sdxl"]
LABELS = {"sd15": "SD 1.5", "sd21": "SD 2.1", "sdxl": "SDXL"}

z0s    = {}   # model_key → tensor (1, C, H, W) float32
stats  = {}   # model_key → dict
images = {}   # model_key → (PIL generada, PIL reconstruida)

missing = []
for key in MODELS:
    d = os.path.join(DATA_ROOT, key)
    if not os.path.isdir(d):
        missing.append(key)
        print(f"  ✗ {key}: carpeta no encontrada en {d}")
        continue

    z0s[key]   = torch.load(os.path.join(d, "z0.pt"), map_location="cpu")
    with open(os.path.join(d, "stats.json")) as f:
        stats[key] = json.load(f)
    images[key] = (
        Image.open(os.path.join(d, "imagen_generada.png")),
        Image.open(os.path.join(d, "imagen_reconstruida.png")),
    )
    shape = tuple(z0s[key].shape)
    print(f"  ✓ {key:6}  z0={shape}  dim={z0s[key].numel():,}")

if missing:
    print(f"\nATENCIÓN: faltan {missing}. Ejecuta fase1_modelos.ipynb para esos modelos.")
else:
    AVAILABLE = MODELS
    print("\nTodos los modelos cargados correctamente.")

---
## 2 — Estadísticas comparativas del espacio latente

In [ ]:
# Estadísticas descriptivas comparativas del espacio latente z₀.
# La norma L2 crece monótonamente (SD1.5→SD2.1→SDXL) aunque la desviación típica
# decrece: coherente con E[‖z‖] ≈ √(N·(μ²+σ²)) para distribuciones gaussianas.
# El MAE del ciclo encode→decode disminuye progresivamente (4.00→2.64→1.80),
# indicando que los modelos más recientes tienen VAEs de mayor capacidad.
available = [k for k in MODELS if k in z0s]

hdr = f"{'Modelo':<8} {'Shape':<22} {'Dim total':>10} {'Media':>10} {'Std':>10} {'Norma L2':>10} {'MAE VAE':>10}"
print(hdr)
print("-" * len(hdr))

for key in available:
    z = z0s[key].float()
    s = stats[key]
    print(
        f"{key:<8} "
        f"{str(tuple(z.shape)):<22} "
        f"{z.numel():>10,} "
        f"{z.mean().item():>10.4f} "
        f"{z.std().item():>10.4f} "
        f"{z.norm().item():>10.2f} "
        f"{s['vae_mae']:>10.2f}"
    )

---
## 3 — Visualización de canales latentes

In [ ]:
# Visualización de los cuatro canales del tensor z₀ para cada modelo.
# Los canales latentes no tienen interpretación semántica directa (no corresponden
# a colores RGB ni a frecuencias concretas), pero su estructura espacial revela
# propiedades del VAE: en SD 1.5 se observa complementariedad entre pares de
# canales; en SDXL aparecen patrones espaciales más finos, coherentes con su
# menor MAE y mayor capacidad de codificación del autoencoder.
available = [k for k in MODELS if k in z0s]
n_rows    = len(available)

fig, axes = plt.subplots(n_rows, 4, figsize=(14, 4 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]   # asegurar shape (N, 4)

for row, key in enumerate(available):
    z_np = z0s[key].squeeze(0).numpy()   # (4, H, W)
    vmin, vmax = z_np.min(), z_np.max()
    for ch in range(4):
        ax = axes[row, ch]
        im = ax.imshow(z_np[ch], cmap="RdBu", vmin=vmin, vmax=vmax,
                       interpolation="nearest")
        ax.set_title(f"{LABELS[key]} — ch {ch}", fontsize=9)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Canales del espacio latente z₀ (mismo prompt, semilla 42)", fontsize=12)
plt.tight_layout()
out = os.path.join(VIS_ROOT, "fase1_latent_channels.png")
plt.savefig(out, dpi=150)
plt.show()
print(f"Guardado: {out}")

---
## 4 — Comparativa de imágenes generadas y reconstruidas

In [ ]:
available = [k for k in MODELS if k in images]
n_rows    = len(available)

fig, axes = plt.subplots(n_rows, 3, figsize=(13, 5 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

for row, key in enumerate(available):
    gen, rec = images[key]
    diff = np.abs(
        np.array(gen).astype(float) - np.array(rec).astype(float)
    )
    axes[row, 0].imshow(gen)
    axes[row, 0].set_title(f"{LABELS[key]} — Generada")
    axes[row, 1].imshow(rec)
    axes[row, 1].set_title(f"{LABELS[key]} — Reconstruida")
    axes[row, 2].imshow((diff * 5).clip(0, 255).astype(np.uint8), cmap="hot")
    axes[row, 2].set_title(f"Diferencia ×5  (MAE={stats[key]['vae_mae']:.1f})")
    for ax in axes[row]: ax.axis("off")

plt.suptitle("Round-trip VAE por modelo", fontsize=13)
plt.tight_layout()
out = os.path.join(VIS_ROOT, "fase1_roundtrip_comparativa.png")
plt.savefig(out, dpi=150)
plt.show()
print(f"Guardado: {out}")

---
## 5 — Resumen y exportación

In [ ]:
import datetime

available = [k for k in MODELS if k in stats]

summary = {
    "fase":               1,
    "fecha_comparativa":  datetime.datetime.now().isoformat(),
    "modelos_analizados": available,
    "modelos":            {k: stats[k] for k in available},
}

out_path = os.path.join(DATA_ROOT, "fase1_summary.json")
os.makedirs(DATA_ROOT, exist_ok=True)
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("Resumen guardado en:", out_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))